# Task 10 &nbsp;|&nbsp; Frozen feature extractor + Optuna hyper-parameter search

**Dr. Sattar's request.** *"Use only Feature Extraction Layers of Pre-trained model, remove
classification layers, and add your own classification layers. Implement automated
hyper-parameter tuning using Optuna of Pytorch."*

This notebook does exactly that, and nothing else:

1. Loads the ImageNet-pretrained backbone and **deletes its classification layers**.
2. **Freezes** every remaining parameter, so the backbone is a fixed feature extractor.
3. Attaches **our own classification head**, whose shape is not fixed in advance.
4. Uses **Optuna** to choose that head's architecture and optimisation settings automatically.

### The three constraints this notebook is built around

Dr. Chattha set three conditions on this work. Each one is enforced by code in this notebook, not
by good intentions, and each has an audit cell that prints the evidence.

| | Constraint | Where it is enforced |
|---|---|---|
| **(a)** | *"Optuna must never see test or out-of-fold performance. Keep the search inside the training portion, using a nested or inner validation split, and report the search space and trial budget in the paper."* | §5 nested inner CV, **Audit A8** |
| **(b)** | *"Keep the fingerprint protocol intact. If anything in the tuning work changes the data pipeline or the folds, the fingerprint changes, and Task 5 must be rerun so that the comparison remains valid."* | §1.2 refuses to build a cache; §3 **hard-asserts** the fingerprint, **Audit A9** |
| **(c)** | *"Treat the frozen-backbone variant as an ablation, not a replacement for the baseline. Pretrained features versus full fine-tuning is a useful ablation row for Milestone 8, but our contribution is uncertainty-aware and agentic inference, so please do not chase a higher number at the cost of that story."* | §7 reports the Task 04-01 baseline as the headline row; every output is named `task10_*_ablation_*`; §8 states the uncertainty cost |

### How (b) is made structural rather than hopeful

Task 04-01 and Task 05-01 could rebuild the nodule cache from raw DICOM if none was attached.
**Task 10 cannot.** It requires the already-built `lidc_nodules_cache_v2.npz` and stops if it is
absent. Since the cache is fixed, the nodules are fixed; since `PROTOCOL_KEYS` are copied verbatim
from Task 04-01, the folds are fixed; so the fingerprint is fixed. The notebook then checks it
against the expected value and refuses to continue on a mismatch. There is no code path in which
this notebook silently changes the data pipeline.

That also means the frozen-backbone predictions are **paired** with the Task 04-01 and Task 05-01
predictions on the same nodules in the same folds, so DeLong and McNemar remain valid (§7.2).

### What this notebook deliberately does not do

It does not claim the frozen variant as the new baseline, it does not re-select the protocol, and
it does not tune anything against the test folds. If the tuned frozen model beats full fine-tuning,
that is an ablation row. If it loses, that is the same ablation row with a different number, and it
is reported either way.

## 1 &nbsp;Configuration

`PROTOCOL_KEYS` and their values are copied **verbatim** from Task 04-01. They are what the
fingerprint hashes. Everything new in this notebook lives under the frozen-feature and Optuna
blocks, which are deliberately *not* protocol keys: changing the search space changes the model,
not the data, so it must not change the fingerprint.

In [ ]:
import os, sys, json, math, time, random, warnings, glob, zipfile, shutil, hashlib
from pathlib import Path
warnings.filterwarnings("ignore")

BUILD_VERSION = "2.0-mm-throughplane"    # must match the cache Task 04-01 / 05-01 produced

CFG = dict(
    SEED = 42,

    # ---------- which backbone to freeze ----------
    # "resnet18" freezes the Task 04-01 arm's backbone; "vit_tiny" freezes the Task 05-01 arm's.
    MODEL = "resnet18",
    PRETRAINED = True,        # a frozen RANDOM backbone would be a different (and pointless) study

    # ---------- data ----------
    # Task 10 consumes the cache, it never builds one. See the note in the header.
    CACHE_NAME = "lidc_nodules_cache_v2.npz",
    WORK       = "/kaggle/working",

    # ---------- nodule extraction (PROTOCOL - hashed - copied verbatim from Task 04-01) ----------
    N_PATIENTS   = 300,
    MIN_READERS  = 3,
    DROP_SCORE_3 = True,
    MATCH_MM     = 6.0,
    PATCH_MM     = 48.0,
    IMG_SIZE     = 224,
    N_SLICES     = 3,
    SLICE_MM     = 2.5,
    HU_WINDOW    = (-1000.0, 400.0),

    # ---------- patient-level cross-validation (PROTOCOL - hashed) ----------
    N_FOLDS = 5,
    VAL_FRAC_OF_TRAIN = 0.15,

    # ---------- frozen feature extraction ----------
    N_AUG = 8,             # precomputed augmented copies of each TRAINING nodule (see 4.2)
    FEATURE_BATCH = 64,

    # ---------- Optuna (NOT protocol - touches neither the data nor the folds) ----------
    N_TRIALS = 40,         # per outer fold; reported trial budget is N_TRIALS x N_FOLDS
    INNER_FOLDS = 3,       # nested inner CV, inside the training portion only
    HEAD_MAX_EPOCHS = 80,
    HEAD_PATIENCE = 12,
    OPTUNA_TIMEOUT_S = None,   # set e.g. 1800 to cap the search by wall clock instead

    # ---------- evaluation (copied from Task 04-01 so the numbers are comparable) ----------
    TARGET_SPECIFICITIES = [0.90, 0.95, 0.99],
    N_BOOTSTRAP = 1000,
    N_ECE_BINS = 15,
    ALLOW_CPU_FALLBACK = True,
)

CFG["OUT_DIR"] = f"{CFG['WORK']}/task10_outputs"
Path(CFG["OUT_DIR"]).mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["benign", "malignant"]
IS_SMOKE = False

PROTOCOL_KEYS = ["SEED", "N_PATIENTS", "MIN_READERS", "DROP_SCORE_3", "MATCH_MM", "PATCH_MM",
                 "IMG_SIZE", "N_SLICES", "SLICE_MM", "HU_WINDOW", "N_FOLDS", "VAL_FRAC_OF_TRAIN"]

# Constraint (b). This is the fingerprint Task 04-01 and Task 05-01 both printed on the v2 cache.
# If this notebook computes anything else, something in the data pipeline moved and the cross-task
# comparison is void - so it stops rather than quietly reporting a number.
EXPECTED_FINGERPRINT = "5922c68330c537b8"
ENFORCE_FINGERPRINT  = True    # leave True; set False only to explore a deliberately new protocol

print(f"Task 10  |  build {BUILD_VERSION}  |  backbone {CFG['MODEL']} (frozen)")
print(f"expected fingerprint: {EXPECTED_FINGERPRINT}  (enforced: {ENFORCE_FINGERPRINT})")
print(f"trial budget: {CFG['N_TRIALS']} trials x {CFG['N_FOLDS']} outer folds = "
      f"{CFG['N_TRIALS'] * CFG['N_FOLDS']} trials, "
      f"each scored on {CFG['INNER_FOLDS']}-fold inner CV")

### 1.1 &nbsp;Environment

Optuna is not always present on a fresh Kaggle image, so it is installed if missing. The GPU
preflight is the same one as Task 04-01: it runs a real convolution before any expensive work, so
an `sm_60` P100 fails here in two seconds rather than half an hour in.

A frozen backbone is far cheaper than fine-tuning — the backbone runs **once** per nodule (§4.2),
not once per epoch — so CPU is a perfectly reasonable place to run this notebook.

In [ ]:
try:
    import optuna
except ImportError:
    print("installing optuna ...")
    os.system(f"{sys.executable} -m pip install -q optuna")
    import optuna

import numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib.pyplot as plt
optuna.logging.set_verbosity(optuna.logging.WARNING)   # the per-fold table below is the log

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| optuna", optuna.__version__, "| numpy", np.__version__)

def gpu_preflight():
    """Prove the GPU can run a convolution before the expensive work begins."""
    global DEVICE
    if not torch.cuda.is_available():
        print("No accelerator attached - running on CPU.")
        print("That is fine here: the frozen backbone is evaluated once per nodule, not per epoch.")
        return False
    name = torch.cuda.get_device_name(0)
    cap  = torch.cuda.get_device_capability(0)
    mine = f"sm_{cap[0]}{cap[1]}"
    built = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
    print(f"GPU {name} ({mine}) | torch built for {', '.join(built)}")
    try:
        torch.nn.functional.conv2d(torch.zeros(1, 3, 8, 8, device="cuda"),
                                   torch.zeros(4, 3, 3, 3, device="cuda"))
        torch.cuda.synchronize()
        print("preflight OK - a real convolution ran on the device")
        return True
    except Exception as e:
        print(f"\n{'='*70}\nTHIS GPU CANNOT RUN THIS PYTORCH BUILD\n{'='*70}")
        print(f"  {name} is {mine}; this wheel has {', '.join(built)}")
        print(f"  {type(e).__name__}: {e}")
        print(f"  FIX: Session options -> Accelerator -> 'GPU T4 x2' (sm_75)\n{'='*70}")
        if CFG["ALLOW_CPU_FALLBACK"]:
            DEVICE = torch.device("cpu")
            print("  ALLOW_CPU_FALLBACK is on - continuing on CPU.")
            return False
        raise

GPU_OK = gpu_preflight()
print("device:", DEVICE)

### 1.2 &nbsp;Locating the cache &mdash; and why this notebook will not build one

Constraint (b) says the fingerprint protocol must stay intact. The cheapest way to guarantee that
is to delete the code that could break it: **Task 10 has no cache builder.** It looks for the v2
cache in the attached Kaggle datasets and in the working directory, checks the build version
recorded inside the file, and stops with instructions if it cannot find one.

A cache written by build `1.x` took its neighbour slices at ±1 *slice index* instead of ±2.5 mm.
Loading one here would change the patches, change the fingerprint, and void the comparison, so a
v1 cache is rejected rather than warned about.

**Attach:** `Add Data → Your Datasets →` the dataset holding `lidc_nodules_cache_v2.npz`
(the file Task 04-01 wrote).

In [ ]:
def find_cache(cfg):
    """Return (path, scan_log). Only a cache whose recorded build version matches is accepted."""
    scan, cands = [], []
    for r in ["/kaggle/input", cfg["WORK"], "."]:
        cands += sorted(glob.glob(f"{r}/**/{cfg['CACHE_NAME']}", recursive=True))
        cands += sorted(glob.glob(f"{r}/**/lidc_nodules_cache*.npz", recursive=True))
    for p in dict.fromkeys(cands):                       # de-duplicate, preserve order
        try:
            with np.load(p, allow_pickle=True) as z:
                bc = json.loads(str(z["cfg_json"]))
        except Exception as e:
            scan.append((p, f"unreadable ({type(e).__name__})")); continue
        ver = bc.get("BUILD_VERSION", "1.x (pre-fix)")
        if ver != BUILD_VERSION:
            scan.append((p, f"build {ver} - REJECTED (need {BUILD_VERSION})")); continue
        scan.append((p, f"build {ver} - ACCEPTED"))
        return p, scan
    return None, scan


CACHE, CACHE_SCAN = find_cache(CFG)
for p, why in CACHE_SCAN:
    print(f"  {why:44s} {p}")
if not CACHE_SCAN:
    print("  (no lidc_nodules_cache*.npz found anywhere)")

if CACHE is None:
    raise FileNotFoundError(
        "\n" + "=" * 76 +
        "\nTASK 10 NEEDS THE v2 NODULE CACHE AND WILL NOT REBUILD IT." + "\n" + "=" * 76 +
        f"\nNo '{CFG['CACHE_NAME']}' with build version '{BUILD_VERSION}' was found."
        "\n\nWhy there is no fallback: rebuilding would re-derive the nodule set and the folds,"
        "\nwhich changes the dataset fingerprint. Constraint (b) is that the fingerprint stays"
        "\nintact so Task 5 does not have to be rerun. Refusing to build is how that is"
        "\nguaranteed rather than hoped for."
        "\n\nFIX: Kaggle -> Add Data -> Your Datasets -> the dataset holding"
        "\nlidc_nodules_cache_v2.npz (the file Task 04-01 wrote), then re-run this cell.\n"
        + "=" * 76)

print(f"\nusing cache: {CACHE}")

In [ ]:
with np.load(CACHE, allow_pickle=True) as z:
    PATCHES    = z["patches"]
    LABELS     = z["labels"].astype(np.int64)
    MALIGNANCY = z["malignancy"].astype(np.float32)
    PATIENT_ID = np.asarray([str(v) for v in z["patient_id"]])
    BUILD_CFG  = json.loads(str(z["cfg_json"]))
    META       = pd.DataFrame(json.loads(str(z["meta_json"])))

IS_SMOKE = bool(BUILD_CFG.get("PHANTOM", False))

print(f"patches {PATCHES.shape} {PATCHES.dtype} | {len(LABELS)} nodules, "
      f"{len(np.unique(PATIENT_ID))} patients")
print(f"benign {int((LABELS==0).sum())} | malignant {int((LABELS==1).sum())} "
      f"({100*LABELS.mean():.1f}% positive)")
print(f"cache build version: {BUILD_CFG.get('BUILD_VERSION')}  |  SMOKE: {IS_SMOKE}")

# The protocol values recorded inside the cache must agree with ours, or the folds computed below
# would describe a different dataset than the one actually loaded.
def canon(v):
    """JSON stores HU_WINDOW as a list but CFG holds a tuple, so compare structurally, not by str."""
    if isinstance(v, (list, tuple)):
        return [canon(x) for x in v]
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    return v

drift = {k: (BUILD_CFG.get(k), CFG[k]) for k in PROTOCOL_KEYS
         if k in BUILD_CFG and canon(BUILD_CFG[k]) != canon(CFG[k])}
if drift:
    print("\nPROTOCOL DRIFT between the cache and this notebook:")
    for k, (a, b) in drift.items():
        print(f"  {k}: cache={a!r}  notebook={b!r}")
    raise ValueError("The cache was built under different protocol values (listed above). "
                     "Fix CFG to match the cache, or attach the matching cache.")
print("protocol values recorded in the cache match this notebook exactly")

## 2 &nbsp;Patient-level folds &mdash; byte-identical to Task 04-01

This is the same function, with the same inputs, so it produces the same folds. Nothing here is
new work; it is reproduced so the notebook is self-contained and so the fingerprint can be
recomputed rather than trusted.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def patient_level_folds(labels, groups, n_folds, val_frac, seed):
    idx = np.arange(len(labels))
    outer = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for k, (tr_pool, test_idx) in enumerate(outer.split(idx, labels, groups)):
        k_val = max(2, int(round(1.0 / val_frac)))
        sub_tr, sub_val = next(StratifiedGroupKFold(
            n_splits=k_val, shuffle=True, random_state=seed + 100 + k
        ).split(tr_pool, labels[tr_pool], groups[tr_pool]))
        folds.append(dict(fold=k, train=tr_pool[sub_tr], val=tr_pool[sub_val], test=test_idx))
    return folds


def dataset_fingerprint(labels, groups, folds, cfg):
    """Hash of everything that must match for two runs to be comparable."""
    h = hashlib.sha256()
    h.update(np.asarray(labels, np.int64).tobytes())
    h.update("|".join(map(str, groups)).encode())
    for k in PROTOCOL_KEYS:
        h.update(f"{k}={cfg[k]}".encode())
    fold_of = np.full(len(labels), -1, np.int64)
    for f in folds:
        fold_of[f["test"]] = f["fold"]
    h.update(fold_of.tobytes())
    return h.hexdigest()[:16], fold_of


FOLDS = patient_level_folds(LABELS, PATIENT_ID, CFG["N_FOLDS"],
                            CFG["VAL_FRAC_OF_TRAIN"], CFG["SEED"])
FINGERPRINT, FOLD_OF = dataset_fingerprint(LABELS, PATIENT_ID, FOLDS, CFG)

print(pd.DataFrame([
    dict(fold=f["fold"],
         train_nod=len(f["train"]), val_nod=len(f["val"]), test_nod=len(f["test"]),
         train_pat=len(np.unique(PATIENT_ID[f["train"]])),
         val_pat=len(np.unique(PATIENT_ID[f["val"]])),
         test_pat=len(np.unique(PATIENT_ID[f["test"]])),
         test_pct_malig=round(100 * float(LABELS[f["test"]].mean()), 1))
    for f in FOLDS]).to_string(index=False))
print(f"\nDATASET FINGERPRINT: {FINGERPRINT}")

### Audit A9 &nbsp;|&nbsp; Constraint (b) &mdash; the fingerprint is unchanged

If this assertion fires, the tuning work has moved the data pipeline or the folds, and by
Dr. Chattha's condition Task 5 would have to be rerun before any comparison is quoted. The
notebook stops here so that cannot happen by accident.

In [ ]:
print(f"A9.1  computed fingerprint : {FINGERPRINT}")
print(f"A9.2  expected fingerprint : {EXPECTED_FINGERPRINT}   (Task 04-01 and Task 05-01, v2 cache)")

MATCH = (FINGERPRINT == EXPECTED_FINGERPRINT)
print(f"A9.3  match                : {MATCH}")

if not MATCH:
    msg = (f"\n{'='*76}\nFINGERPRINT MISMATCH - CONSTRAINT (b) VIOLATED\n{'='*76}\n"
           f"  computed {FINGERPRINT}, expected {EXPECTED_FINGERPRINT}\n\n"
           "  Something about the data pipeline or the fold assignment differs from the runs that\n"
           "  produced the Task 04-01 and Task 05-01 results. Paired comparisons (DeLong, McNemar)\n"
           "  against those runs are NOT valid until this is resolved.\n\n"
           "  Most likely causes, in order:\n"
           "    1. a different cache file is attached (check the path printed in 1.2)\n"
           "    2. a PROTOCOL_KEYS value was edited in this notebook\n"
           "    3. the cache itself was rebuilt under a changed protocol\n\n"
           "  Per constraint (b): if the protocol genuinely needs to change, Task 5 must be rerun\n"
           "  on the new protocol before the comparison table in section 7 means anything.\n"
           f"{'='*76}")
    if ENFORCE_FINGERPRINT:
        raise AssertionError(msg)
    print(msg + "\n  ENFORCE_FINGERPRINT is off - continuing, but results are NOT comparable.")
else:
    print("\nA9.4  The nodules, the labels and the five test folds are identical to those used by")
    print("      Task 04-01 and Task 05-01. Paired statistics against those runs are valid, and")
    print("      Task 5 does not need to be rerun.")

## 3 &nbsp;Removing the classification layers and freezing what is left

This is the literal content of Dr. Sattar's request, so it is worth being precise about what
"feature extraction layers" means for each backbone.

**ResNet-18.** The network is a stem, four residual stages, a global average pool, and a single
`fc: Linear(512 → 1000)` that maps pooled features to ImageNet classes. That `fc` *is* the
classification layer, and it is the only one. We keep everything up to and including the pool, and
replace `fc` with `nn.Identity()`. What comes out is a **512-dimensional** feature vector.

**ViT-tiny.** The transformer blocks produce token embeddings; `head: Linear(192 → 1000)` maps the
pooled class token to ImageNet classes. `timm.create_model(..., num_classes=0)` removes that head
and returns the pooled embedding directly — a **192-dimensional** feature vector.

### Freezing is two things, not one

Setting `requires_grad = False` stops the optimiser updating the weights. It does **not** stop
BatchNorm layers updating their running mean and variance, because those are buffers, not
parameters — they are updated by the forward pass whenever the module is in training mode. A
ResNet-18 has 20 BatchNorm layers. If they keep adapting, the "frozen" backbone is not frozen: its
outputs drift with the data it sees, the precomputed features silently go stale, and the ablation
no longer measures what it claims to.

So `FrozenBackbone` overrides `train()` to be a no-op that always puts the module in `eval()` mode.
Calling `.train()` on it, directly or through a parent module, cannot switch BatchNorm back on.
Audit A7 verifies both halves of this by checking the buffers before and after a forward pass.

In [ ]:
import torchvision

# ImageNet statistics, identical to Task 04-01 / 05-01. Defined here because Audit A7 below already
# needs them to push a real patch through the backbone.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

FEATURE_DIMS = {"resnet18": 512, "vit_tiny": 192}

class FrozenBackbone(nn.Module):
    """Pretrained feature extractor with the classification layers removed and everything frozen.

    train() is overridden to a no-op so that no parent module, and no stray .train() call inside a
    training loop, can put the BatchNorm layers back into updating mode.
    """

    def __init__(self, name="resnet18", pretrained=True):
        super().__init__()
        self.name = name
        if name == "resnet18":
            net = torchvision.models.resnet18(
                weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            self.removed = f"fc: Linear({net.fc.in_features} -> {net.fc.out_features})"
            net.fc = nn.Identity()          # <- the classification layer, deleted
            self.body = net
            self.out_dim = FEATURE_DIMS["resnet18"]
        elif name == "vit_tiny":
            import timm
            probe = timm.create_model("vit_tiny_patch16_224", pretrained=False)
            self.removed = f"head: Linear({probe.head.in_features} -> {probe.head.out_features})"
            del probe
            # num_classes=0 is timm's documented way to drop the classifier and return features
            self.body = timm.create_model("vit_tiny_patch16_224", pretrained=pretrained,
                                          num_classes=0, img_size=CFG["IMG_SIZE"])
            self.out_dim = FEATURE_DIMS["vit_tiny"]
        else:
            raise ValueError(f"unknown backbone {name!r}; expected 'resnet18' or 'vit_tiny'")

        for p in self.body.parameters():
            p.requires_grad_(False)
        super().train(False)                # start in eval mode

    def train(self, mode=True):
        """Deliberately ignores `mode`. A frozen extractor is always in eval mode."""
        return super().train(False)

    @torch.no_grad()
    def forward(self, x):
        return self.body(x).flatten(1)


BACKBONE = FrozenBackbone(CFG["MODEL"], CFG["PRETRAINED"]).to(DEVICE)
FEAT_DIM = BACKBONE.out_dim

n_total = sum(p.numel() for p in BACKBONE.parameters())
n_train = sum(p.numel() for p in BACKBONE.parameters() if p.requires_grad)
print(f"backbone            : {CFG['MODEL']} (pretrained={CFG['PRETRAINED']})")
print(f"classification layer: {BACKBONE.removed}  -> REMOVED")
print(f"feature dimension   : {FEAT_DIM}")
print(f"parameters          : {n_total:,} total, {n_train:,} trainable")

### Audit A7 &nbsp;|&nbsp; The backbone really is frozen

Four checks. A7.1 and A7.2 are about gradients; A7.3 is the BatchNorm buffer check described above,
and is the one that catches the common mistake; A7.4 confirms that the same input gives bit-identical
features on two separate calls, which is what makes precomputing them in §4 legitimate.

In [ ]:
print("A7.1  No backbone parameter requires a gradient")
req = [n for n, p in BACKBONE.named_parameters() if p.requires_grad]
print(f"      parameters with requires_grad=True: {len(req)}")
assert len(req) == 0, f"these are still trainable: {req[:5]}"

print("\nA7.2  The classification layer is gone")
if CFG["MODEL"] == "resnet18":
    print(f"      body.fc is now {type(BACKBONE.body.fc).__name__}")
    assert isinstance(BACKBONE.body.fc, nn.Identity)
else:
    print(f"      timm num_classes=0; head is {type(getattr(BACKBONE.body, 'head', None)).__name__}")
probe_out = BACKBONE(torch.zeros(2, 3, CFG["IMG_SIZE"], CFG["IMG_SIZE"], device=DEVICE))
print(f"      forward output shape {tuple(probe_out.shape)} - a feature vector, not class logits")
assert probe_out.shape[1] == FEAT_DIM

# A real batch of LIDC patches, normalised the way the training pipeline will normalise them.
_raw = np.stack([p[:3] if p.shape[0] >= 3 else np.repeat(p, 3, 0) for p in PATCHES[:8]])
REAL_BATCH = torch.from_numpy(_raw.astype(np.float32) / 255.0).to(DEVICE)
REAL_BATCH = (REAL_BATCH - IMAGENET_MEAN.to(DEVICE)) / IMAGENET_STD.to(DEVICE)

print("\nA7.3  BatchNorm running statistics do not move, even after .train() is called")
bns = [m for m in BACKBONE.modules() if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d))]
print(f"      BatchNorm layers in this backbone: {len(bns)}")
BACKBONE.train()                                      # the call that would normally unfreeze them
print(f"      after BACKBONE.train(): module.training = {BACKBONE.training} "
      f"(a frozen extractor ignores the request)")
assert BACKBONE.training is False
if bns:
    before = [(m.running_mean.clone(), m.running_var.clone(), m.num_batches_tracked.clone())
              for m in bns]
    _ = BACKBONE(REAL_BATCH)
    moved = sum(1 for m, (mu, var, nb) in zip(bns, before)
                if not (torch.equal(m.running_mean, mu) and torch.equal(m.running_var, var)
                        and torch.equal(m.num_batches_tracked, nb)))
    print(f"      BatchNorm layers whose running stats changed after a forward pass: {moved}")
    assert moved == 0, "the backbone is adapting to LIDC data - it is not frozen"
else:
    print("      (none - ViT uses LayerNorm, which has no running statistics to drift)")

print("\nA7.4  The same input gives identical features twice (so caching them is safe)")
a, b = BACKBONE(REAL_BATCH), BACKBONE(REAL_BATCH)
print(f"      max |f(x) - f(x)| over two calls: {float((a - b).abs().max()):.3e}")
assert torch.equal(a, b), "features are not deterministic - do not cache them"
print("\nAudit A7 passed: the backbone is a fixed function of its input.")

## 4 &nbsp;Precomputing features

### 4.1 &nbsp;Why this is the step that makes an honest search affordable

Constraint (a) requires a **nested** search: for each of the 5 outer folds, Optuna must be scored on
an inner validation split carved from that fold's training portion, never on the test fold. Done
naively that is

&nbsp;&nbsp;&nbsp;&nbsp;`5 outer folds × 40 trials × 3 inner folds × 80 epochs` of full-network forward passes,

which is roughly two orders of magnitude more compute than Task 04-01 used, and is simply not
runnable in a Kaggle session. The usual response is to cut corners on the protocol — fewer trials,
or, much worse, tuning against the out-of-fold predictions. That is the exact failure Dr. Chattha
warned about.

A frozen backbone removes the problem instead of trading against it. Because the backbone is a
fixed function (Audit A7.4), its output for a given image never changes, so it can be evaluated
**once** and reused by every epoch of every inner fold of every trial. The search then trains only
the head, on cached vectors of 512 (or 192) floats. What was hours becomes minutes, and the honest
nested protocol is the cheap option rather than the expensive one.

### 4.2 &nbsp;Keeping augmentation, despite the caching

The obvious objection to caching is that it appears to kill data augmentation: if each nodule has
one stored feature vector, the head never sees a flipped or rotated version, and the ablation would
then confound "frozen vs fine-tuned" with "augmented vs not".

So instead of caching one vector per nodule, we cache `N_AUG = 8` **augmented copies** plus one
clean copy, using the identical augmentation from Task 04-01 (flips, 90° rotations, mild
intensity jitter). The head trains on 9 views per nodule; augmentation survives. It is a fixed
sample of the augmentation distribution rather than a fresh draw each epoch, which is a real but
small difference, and it is stated here rather than buried.

Augmented copies are generated for **all** nodules, with a per-nodule seed, before any fold is
considered. Augmentation is a label-free, index-local operation, so producing it globally leaks
nothing; which rows are then used for training is decided entirely by the fold indices in §5.

In [ ]:
def to_chw3(x):
    """uint8 patch -> float CHW in [0,1] with exactly 3 channels. Same rule as Task 04-01."""
    x = x.astype(np.float32) / 255.0
    if x.shape[0] == 1:
        x = np.repeat(x, 3, axis=0)
    elif x.shape[0] > 3:
        mid = x.shape[0] // 2
        x = x[mid - 1: mid + 2]
    return x


def augment(x, rng):
    """The Task 04-01 training augmentation, verbatim, so the ablation isolates the backbone."""
    if rng.random() < .5: x = np.flip(x, axis=2)
    if rng.random() < .5: x = np.flip(x, axis=1)
    x = np.ascontiguousarray(np.rot90(x, int(rng.integers(0, 4)), axes=(1, 2)))
    if rng.random() < .3:
        x = np.clip(x * rng.uniform(.93, 1.07) + rng.uniform(-.04, .04), 0, 1)
    return np.ascontiguousarray(x)


@torch.no_grad()
def extract_features(backbone, patches, aug_round, seed, batch=64):
    """Features for every row of `patches`. aug_round=None gives the clean view; an integer gives
    one augmented view, seeded per (round, nodule) so the result is reproducible."""
    n = len(patches)
    out = np.zeros((n, backbone.out_dim), np.float32)
    mean, std = IMAGENET_MEAN.to(DEVICE), IMAGENET_STD.to(DEVICE)
    for s in range(0, n, batch):
        e = min(s + batch, n)
        buf = []
        for j in range(s, e):
            x = to_chw3(patches[j])
            if aug_round is not None:
                x = augment(x, np.random.default_rng(seed + 1000 * aug_round + j))
            buf.append(x)
        xb = torch.from_numpy(np.stack(buf)).to(DEVICE)
        out[s:e] = backbone((xb - mean) / std).float().cpu().numpy()
    return out


t0 = time.time()
FEAT_CLEAN = extract_features(BACKBONE, PATCHES, None, CFG["SEED"], CFG["FEATURE_BATCH"])
FEAT_AUG = np.stack([extract_features(BACKBONE, PATCHES, r, CFG["SEED"], CFG["FEATURE_BATCH"])
                     for r in range(CFG["N_AUG"])])
took = time.time() - t0

print(f"clean features    : {FEAT_CLEAN.shape}")
print(f"augmented features: {FEAT_AUG.shape}   ({CFG['N_AUG']} views per nodule)")
print(f"extraction took   : {took/60:.1f} min for {(1 + CFG['N_AUG']) * len(PATCHES):,} "
      f"backbone passes on {DEVICE.type.upper()}")
print(f"cached size       : {(FEAT_CLEAN.nbytes + FEAT_AUG.nbytes) / 1e6:.1f} MB")
print(f"\nEvery Optuna trial from here on reads these arrays; the backbone is not touched again.")

# A frozen backbone that produced constant or degenerate features would make the whole study
# meaningless, and would not show up as an error anywhere else.
sd = FEAT_CLEAN.std(axis=0)
print(f"feature std across nodules: min {sd.min():.4f}, median {np.median(sd):.4f}, max {sd.max():.4f}")
assert sd.max() > 1e-3, "the backbone produced essentially constant features"
print(f"dead features (std < 1e-6): {int((sd < 1e-6).sum())} of {FEAT_DIM}")

## 5 &nbsp;Our own classification layers

Section 3 deleted the pretrained classifier. This is what replaces it — and note that its *shape*
is a search variable, not a constant. Optuna chooses the depth, the width, the activation, whether
to use BatchNorm, and how much dropout:

```
features (512 or 192)
  └─ [ Linear(d → hidden) → (BatchNorm1d) → ReLU|GELU → Dropout(p) ] × n_layers
       └─ Dropout(p) if n_layers == 0
            └─ Linear(d → 2)
```

`n_layers = 0` collapses to a regularised linear probe on frozen features, which is the classical
transfer-learning baseline and a sensible thing for the search to be able to pick.

**Standardisation is fitted inside the head, on training rows only.** Frozen ResNet features come
out of a ReLU, so they are non-negative and very unevenly scaled; without standardisation the
search wastes most of its budget on learning rates. The mean and standard deviation are computed
from the training rows of whichever split is currently active — inner-training rows during the
search, outer-training rows for the final fit — and never from validation or test rows. This is the
same preprocessing-provenance rule audited as A2 in Task 04-01, applied one level deeper.

There is always at least one `Dropout` before the final `Linear`, so MC Dropout (Task 6) can still
run on this model. Section 8 explains why that is nonetheless a weaker uncertainty estimate than
the fine-tuned network gives.

In [ ]:
from torch.optim import AdamW
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, brier_score_loss)

class ClassifierHead(nn.Module):
    """The classification layers we add in place of the pretrained ones removed in section 3."""

    def __init__(self, in_dim, n_layers=0, hidden=256, p=0.3, act="relu", use_bn=True, n_classes=2):
        super().__init__()
        acts = {"relu": nn.ReLU, "gelu": nn.GELU}
        layers, d = [], in_dim
        for _ in range(n_layers):
            layers.append(nn.Linear(d, hidden))
            if use_bn:
                layers.append(nn.BatchNorm1d(hidden))
            layers += [acts[act](), nn.Dropout(p)]
            d = hidden
        if n_layers == 0:
            # guarantee at least one stochastic layer, so Task 6's MC Dropout still has something
            # to sample even when the search picks a plain linear probe
            layers.append(nn.Dropout(p))
        layers.append(nn.Linear(d, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_head(Xtr, ytr, Xva, yva, params, cfg, seed):
    """Fit a head on precomputed features. Returns (state_dict, best_val_auroc, best_epoch, scaler).

    The feature scaler is fitted on Xtr only; Xva is transformed with it, never used to fit it.
    """
    set_seed(seed)
    mu = Xtr.mean(axis=0, keepdims=True)
    sd = Xtr.std(axis=0, keepdims=True) + 1e-6
    xtr = torch.from_numpy((Xtr - mu) / sd).float().to(DEVICE)
    xva = torch.from_numpy((Xva - mu) / sd).float().to(DEVICE)
    ttr = torch.from_numpy(np.asarray(ytr, np.int64)).to(DEVICE)

    head = ClassifierHead(Xtr.shape[1], params["n_layers"], params["hidden"], params["dropout"],
                          params["act"], params["use_bn"]).to(DEVICE)

    cnt = np.bincount(np.asarray(ytr, np.int64), minlength=2).astype(np.float32)
    w = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=w)                 # same class weighting as Task 04-01
    opt = AdamW(head.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])

    n, bs = len(xtr), params["batch_size"]
    g = torch.Generator(device="cpu").manual_seed(seed)
    best, best_ep, best_state, bad = -np.inf, -1, None, 0

    for ep in range(1, cfg["HEAD_MAX_EPOCHS"] + 1):
        head.train()
        perm = torch.randperm(n, generator=g)
        for s in range(0, n, bs):
            b = perm[s:s + bs]
            if len(b) < 2:                     # BatchNorm1d needs >1 row in training mode
                continue
            opt.zero_grad(set_to_none=True)
            criterion(head(xtr[b]), ttr[b]).backward()
            opt.step()

        head.eval()
        with torch.no_grad():
            pv = torch.softmax(head(xva).float(), 1)[:, 1].cpu().numpy()
        auc = roc_auc_score(yva, pv) if len(np.unique(yva)) > 1 else np.nan

        if np.isfinite(auc) and auc > best + 1e-5:
            best, best_ep, bad = float(auc), ep, 0
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["HEAD_PATIENCE"]:
                break

    return best_state, best, best_ep, (mu, sd)


@torch.no_grad()
def predict_head(state, params, in_dim, scaler, X):
    mu, sd = scaler
    head = ClassifierHead(in_dim, params["n_layers"], params["hidden"], params["dropout"],
                          params["act"], params["use_bn"]).to(DEVICE)
    head.load_state_dict(state); head.eval()
    x = torch.from_numpy((X - mu) / sd).float().to(DEVICE)
    return torch.softmax(head(x).float(), 1)[:, 1].cpu().numpy()


def training_rows(indices):
    """Feature matrix for training on `indices`: the clean view plus all N_AUG augmented views."""
    X = np.concatenate([FEAT_CLEAN[indices]] + [FEAT_AUG[r][indices] for r in range(CFG["N_AUG"])])
    y = np.tile(LABELS[indices], CFG["N_AUG"] + 1)
    return X, y


print(f"head input dimension: {FEAT_DIM}")
demo = ClassifierHead(FEAT_DIM, 2, 256, 0.3)
print(f"example architecture (n_layers=2, hidden=256):\n{demo}")
print(f"trainable parameters in that example: "
      f"{sum(p.numel() for p in demo.parameters()):,}  "
      f"(vs 11.2M for the fully fine-tuned ResNet-18)")

## 6 &nbsp;Automated hyper-parameter search with Optuna

### 6.1 &nbsp;The search space and the trial budget

Constraint (a) asks for both of these to be reportable, so they are defined in one place and
written to `task10_*_search_space.json` for the paper.

| Hyper-parameter | Range | Sampling |
|---|---|---|
| `n_layers` | 0, 1, 2 | categorical |
| `hidden` | 64, 128, 256, 512 | categorical (ignored when `n_layers = 0`) |
| `dropout` | 0.0 – 0.6 | uniform |
| `use_bn` | True / False | categorical (ignored when `n_layers = 0`) |
| `act` | ReLU / GELU | categorical (ignored when `n_layers = 0`) |
| `lr` | 1e-4 – 3e-2 | log-uniform |
| `weight_decay` | 1e-6 – 1e-1 | log-uniform |
| `batch_size` | 32, 64, 128 | categorical |

Epochs are not searched: every configuration is trained for up to `HEAD_MAX_EPOCHS = 80` with early
stopping (patience 12) on the inner validation AUROC, so each trial gets its own best epoch for
free. The sampler is TPE with a per-fold seed, and a median pruner stops a trial once its running
inner-fold mean is clearly behind.

**Budget: 40 trials per outer fold × 5 outer folds = 200 trials, each evaluated on 3 inner folds
= 600 head fits.** Because the features are cached, that is minutes rather than days.

### 6.2 &nbsp;Why the search is nested, and what that buys

For outer fold *k*, the search is scored **only** on inner splits of `fold[k]["train"]`:

```
fold k:   [ ------------ train ------------ ] [ val ] [ ---- test ---- ]
               |                                 |            |
               |  inner 3-fold CV, grouped       |            |
               |  by patient -> the Optuna       |            |
               |  objective                      |            |
               |                                 |            |
          Optuna sees this                 early stopping   scored once,
                                           for the final    after the study
                                           fit only         has finished
```

Three separate things are kept apart:

- **Optuna's objective** is the mean AUROC over 3 patient-grouped inner folds of the training
  portion. It is the only signal the sampler ever receives.
- **The outer validation split** is used exactly as Task 04-01 used it — to pick the stopping epoch
  of the final fit — and never to compare configurations.
- **The test fold** is scored once, after the study for that fold is complete.

Hyper-parameters are therefore selected *per outer fold*, from that fold's training data alone.
Five folds can and generally will choose five different configurations; they are listed in §6.4.
That is what makes the out-of-fold AUROC in §7 an honest estimate rather than an optimistic one:
no test nodule influenced any choice made about the model that predicts it.

`TOUCHED_BY_SEARCH` records every cache index read during any trial, and Audit A8 checks it against
the fold definitions afterwards.

In [ ]:
SEARCH_SPACE = {
    "n_layers":     {"type": "categorical", "choices": [0, 1, 2]},
    "hidden":       {"type": "categorical", "choices": [64, 128, 256, 512],
                     "note": "unused when n_layers = 0"},
    "dropout":      {"type": "float", "low": 0.0, "high": 0.6},
    "use_bn":       {"type": "categorical", "choices": [True, False],
                     "note": "unused when n_layers = 0"},
    "act":          {"type": "categorical", "choices": ["relu", "gelu"],
                     "note": "unused when n_layers = 0"},
    "lr":           {"type": "float", "low": 1e-4, "high": 3e-2, "log": True},
    "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-1, "log": True},
    "batch_size":   {"type": "categorical", "choices": [32, 64, 128]},
}

def suggest_params(trial):
    n_layers = trial.suggest_categorical("n_layers", SEARCH_SPACE["n_layers"]["choices"])
    p = dict(
        n_layers     = n_layers,
        dropout      = trial.suggest_float("dropout", 0.0, 0.6),
        lr           = trial.suggest_float("lr", 1e-4, 3e-2, log=True),
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-1, log=True),
        batch_size   = trial.suggest_categorical("batch_size", [32, 64, 128]),
    )
    # Only suggest the layer-shape parameters when there are layers to shape. Optuna handles this
    # conditional space natively; suggesting them unconditionally would make the sampler waste
    # budget modelling dimensions that cannot affect the objective.
    if n_layers > 0:
        p["hidden"] = trial.suggest_categorical("hidden", [64, 128, 256, 512])
        p["use_bn"] = trial.suggest_categorical("use_bn", [True, False])
        p["act"]    = trial.suggest_categorical("act", ["relu", "gelu"])
    else:
        p["hidden"], p["use_bn"], p["act"] = 256, True, "relu"   # inert placeholders
    return p


# The untuned reference point: a plain regularised linear probe on frozen features. Comparing this
# against the tuned head is what isolates the contribution of the search itself, as opposed to the
# contribution of freezing the backbone.
DEFAULT_PARAMS = dict(n_layers=0, hidden=256, dropout=0.30, act="relu", use_bn=True,
                      lr=1e-3, weight_decay=1e-4, batch_size=64)

TOUCHED_BY_SEARCH = {k: set() for k in range(CFG["N_FOLDS"])}

print(json.dumps(SEARCH_SPACE, indent=2))
print(f"\nbudget: {CFG['N_TRIALS']} trials x {CFG['N_FOLDS']} folds x {CFG['INNER_FOLDS']} inner folds "
      f"= {CFG['N_TRIALS'] * CFG['N_FOLDS'] * CFG['INNER_FOLDS']} head fits")
print(f"untuned reference: {DEFAULT_PARAMS}")

### 6.3 &nbsp;Running the search

One Optuna study per outer fold. Failed trials are recorded rather than swallowed: a configuration
that cannot train is a legitimate result for the sampler to learn from, but a configuration that
crashes for an unrelated reason should be visible.

In [ ]:
def make_objective(k, fold):
    """Objective for outer fold k: mean AUROC over patient-grouped inner folds of fold['train']."""
    tr_pool = fold["train"]
    inner = list(StratifiedGroupKFold(n_splits=CFG["INNER_FOLDS"], shuffle=True,
                                      random_state=CFG["SEED"] + 200 + k
                                      ).split(tr_pool, LABELS[tr_pool], PATIENT_ID[tr_pool]))

    def objective(trial):
        params, aucs = suggest_params(trial), []
        for i, (a, b) in enumerate(inner):
            itr, iva = tr_pool[a], tr_pool[b]
            TOUCHED_BY_SEARCH[k].update(itr.tolist())
            TOUCHED_BY_SEARCH[k].update(iva.tolist())

            Xtr, ytr = training_rows(itr)                      # clean + augmented views
            state, auc, _, _ = train_head(Xtr, ytr, FEAT_CLEAN[iva], LABELS[iva],
                                          params, CFG, CFG["SEED"] + 1000 * k + i)
            if state is None or not np.isfinite(auc):
                raise optuna.TrialPruned()                     # never learned anything usable
            aucs.append(auc)

            trial.report(float(np.mean(aucs)), i)              # running mean, for the pruner
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(aucs))

    return objective


STUDIES, BEST_PARAMS, TRIAL_ROWS = {}, {}, []
t_search = time.time()

for k, fold in enumerate(FOLDS):
    t0 = time.time()
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=CFG["SEED"] + k, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1),
        study_name=f"task10_{CFG['MODEL']}_fold{k}")
    study.optimize(make_objective(k, fold), n_trials=CFG["N_TRIALS"],
                   timeout=CFG["OPTUNA_TIMEOUT_S"], catch=())

    STUDIES[k] = study
    best = dict(study.best_trial.params)
    best.setdefault("hidden", 256); best.setdefault("use_bn", True); best.setdefault("act", "relu")
    BEST_PARAMS[k] = best

    states = pd.Series([t.state.name for t in study.trials]).value_counts().to_dict()
    for t in study.trials:
        TRIAL_ROWS.append(dict(fold=k, trial=t.number, state=t.state.name,
                               inner_cv_auroc=t.value, **t.params))
    print(f"fold {k}: best inner-CV AUROC {study.best_value:.4f} "
          f"(trial {study.best_trial.number})  {states}  |  {(time.time()-t0)/60:.1f} min")
    print(f"        {best}")

SEARCH_MIN = (time.time() - t_search) / 60
TRIALS_DF = pd.DataFrame(TRIAL_ROWS)
print(f"\nsearch complete: {len(TRIALS_DF)} trials in {SEARCH_MIN:.1f} min on {DEVICE.type.upper()}")

### Audit A8 &nbsp;|&nbsp; Constraint (a) &mdash; Optuna never saw test or out-of-fold data

A8.1 and A8.2 are the substantive checks: the set of nodules the search actually read, accumulated
during the run, is compared against each fold's definition. A8.3 confirms the objective values are
inner-CV scores. A8.4 states the fact that the outer validation split was used only for early
stopping, which is the same role it had in Task 04-01.

In [ ]:
print("A8.1  Every nodule read during the search belongs to that fold's training portion")
ok = True
for k, fold in enumerate(FOLDS):
    touched, train = TOUCHED_BY_SEARCH[k], set(fold["train"].tolist())
    extra = touched - train
    print(f"      fold {k}: {len(touched):4d} nodules read, {len(train):4d} in train, "
          f"{len(extra)} from outside train")
    ok &= (len(extra) == 0)
assert ok, "the search read nodules outside the training portion"

print("\nA8.2  Overlap of the searched nodules with the validation and test splits")
for k, fold in enumerate(FOLDS):
    n_val  = len(TOUCHED_BY_SEARCH[k] & set(fold["val"].tolist()))
    n_test = len(TOUCHED_BY_SEARCH[k] & set(fold["test"].tolist()))
    print(f"      fold {k}: {n_val} validation nodules, {n_test} test nodules")
    assert n_val == 0 and n_test == 0, f"fold {k} leaked into the search"

print("\nA8.3  Objective values are inner-CV AUROC, not test or out-of-fold AUROC")
for k in range(CFG["N_FOLDS"]):
    print(f"      fold {k}: best objective {STUDIES[k].best_value:.4f} "
          f"= mean over {CFG['INNER_FOLDS']} inner folds of train")

print("\nA8.4  The outer validation split chose the stopping epoch, never the hyper-parameters.")
print("      That is the same role it had in Task 04-01, so the two arms are directly comparable.")
print("\nA8.5  Hyper-parameters were selected independently per outer fold:")
sel = pd.DataFrame([dict(fold=k, **BEST_PARAMS[k]) for k in range(CFG["N_FOLDS"])])
print(sel.to_string(index=False))
print("\nAudit A8 passed: the search is nested inside the training portion of every fold.")

## 7 &nbsp;Final fits and out-of-fold predictions

For each outer fold, two heads are now trained on the **full** training portion of that fold:

- **`frozen_default`** — the untuned linear probe (`DEFAULT_PARAMS`), identical for all five folds.
- **`frozen_tuned`** — the configuration Optuna selected *for that fold* in §6.3.

Both stop on the outer validation AUROC, exactly as Task 04-01 did, and both then predict that
fold's test nodules once. Concatenating the five test folds gives one out-of-fold prediction per
nodule, aligned by `cache_index` with the Task 04-01 and Task 05-01 predictions.

Keeping the untuned probe is what makes the Optuna result interpretable. Without it, a difference
against the fine-tuned baseline confounds two changes at once — freezing the backbone, and
searching the head. With it, `frozen_default → frozen_tuned` isolates the search, and
`baseline → frozen_default` isolates the freezing.

In [ ]:
def fit_arm(params_by_fold, tag):
    """Train one head per fold and return pooled out-of-fold probabilities."""
    oof = np.full(len(LABELS), np.nan)
    rows, ckpts = [], {}
    for k, fold in enumerate(FOLDS):
        params = params_by_fold[k]
        Xtr, ytr = training_rows(fold["train"])
        state, val_auc, best_ep, scaler = train_head(
            Xtr, ytr, FEAT_CLEAN[fold["val"]], LABELS[fold["val"]],
            params, CFG, CFG["SEED"] + 50 + k)
        if state is None:
            raise RuntimeError(f"{tag} fold {k}: the head never improved on validation AUROC")

        p_test = predict_head(state, params, FEAT_DIM, scaler, FEAT_CLEAN[fold["test"]])
        oof[fold["test"]] = p_test
        y_test = LABELS[fold["test"]]
        auc = roc_auc_score(y_test, p_test) if len(np.unique(y_test)) > 1 else np.nan

        ck = f"{CFG['OUT_DIR']}/task10_{CFG['MODEL']}_{tag}_head_fold{k}.pt"
        torch.save(dict(head=state, params=params, feature_mean=scaler[0], feature_std=scaler[1],
                        in_dim=FEAT_DIM, backbone=CFG["MODEL"], frozen=True, arm=tag, fold=k,
                        fingerprint=FINGERPRINT, build_version=BUILD_VERSION), ck)
        ckpts[k] = ck
        rows.append(dict(fold=k, val_auroc=val_auc, best_epoch=best_ep, test_auroc=float(auc),
                         n_test=len(y_test)))
        print(f"  {tag} fold {k}: val {val_auc:.4f} (epoch {best_ep:2d}) -> test {auc:.4f}")

    assert np.isfinite(oof).all(), "some nodule never received an out-of-fold prediction"
    return oof, pd.DataFrame(rows), ckpts


print("untuned linear probe on frozen features")
OOF_DEFAULT, FOLDS_DEFAULT, CKPT_DEFAULT = fit_arm(
    {k: DEFAULT_PARAMS for k in range(CFG["N_FOLDS"])}, "default")

print("\nOptuna-selected head, per fold")
OOF_TUNED, FOLDS_TUNED, CKPT_TUNED = fit_arm(BEST_PARAMS, "tuned")

print(f"\nout-of-fold predictions complete for all {len(LABELS)} nodules")

### 7.1 &nbsp;Metrics

These are the Task 04-01 functions, unchanged, so the numbers in the ablation table are computed
the same way as the numbers they are compared against. Confidence intervals resample **patients**,
not nodules, because nodules from one patient are not independent.

In [ ]:
def sensitivity_at_specificity(y, p, target):
    """Highest sensitivity subject to specificity >= target."""
    fpr, tpr, thr = roc_curve(y, p)
    ok = (1 - fpr) >= target
    if not ok.any():
        return np.nan, np.nan, np.nan
    i = int(np.argmax(np.where(ok, tpr, -np.inf)))
    return float(tpr[i]), float(1 - fpr[i]), float(thr[i])


def expected_calibration_error(y, p, n_bins=15):
    conf = np.where(p >= .5, p, 1 - p)
    correct = ((p >= .5).astype(int) == y).astype(float)
    edges, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for i in range(n_bins):
        m = ((conf > edges[i]) & (conf <= edges[i + 1])) if i else \
            ((conf >= edges[i]) & (conf <= edges[i + 1]))
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def core_metrics(y, p, threshold=0.5, cfg=CFG):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    out = dict(n=int(len(y)), accuracy=float(accuracy_score(y, pred)),
               auroc=float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
               auprc=float(average_precision_score(y, p)),
               sensitivity=float(recall_score(y, pred, zero_division=0)),
               specificity=float(tn / (tn + fp)) if (tn + fp) else np.nan,
               precision=float(precision_score(y, pred, zero_division=0)),
               f1=float(f1_score(y, pred, zero_division=0)),
               ece=expected_calibration_error(y, p, cfg["N_ECE_BINS"]),
               brier=float(brier_score_loss(y, p)),
               tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))
    for ts in cfg["TARGET_SPECIFICITIES"]:
        s, _, _ = sensitivity_at_specificity(y, p, ts)
        out[f"sens_at_spec_{int(ts*100)}"] = s
    return out


def bootstrap_ci(y, p, groups, fn, n_boot, seed):
    """Patient-clustered bootstrap: resample patients, keep all of a patient's nodules together."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    by = {g: np.where(groups == g)[0] for g in uniq}
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([by[g] for g in rng.choice(uniq, len(uniq), replace=True)])
        if len(np.unique(y[idx])) > 1:
            try: vals.append(fn(y[idx], p[idx]))
            except Exception: pass
    if not vals:
        return np.nan, np.nan
    v = np.asarray(vals, float)
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))


M_DEFAULT = core_metrics(LABELS, OOF_DEFAULT)
M_TUNED   = core_metrics(LABELS, OOF_TUNED)
for tag, m in [("frozen_default", M_DEFAULT), ("frozen_tuned", M_TUNED)]:
    lo, hi = bootstrap_ci(LABELS, OOF_DEFAULT if tag.endswith("default") else OOF_TUNED,
                          PATIENT_ID, roc_auc_score, CFG["N_BOOTSTRAP"], CFG["SEED"])
    m["auroc_ci"] = [lo, hi]
    print(f"{tag:16s} AUROC {m['auroc']:.4f} [{lo:.3f}-{hi:.3f}]  "
          f"ECE {m['ece']:.4f}  FN {m['fn']}  sens@spec90 {m['sens_at_spec_90']:.3f}")

### 7.2 &nbsp;The ablation table

Constraint (c): **the baseline is the headline row.** The frozen variants are ablation rows that
describe what pretrained features alone can do; they are not candidates to replace Task 04-01.

The comparison is paired — same nodules, same folds, fingerprint `5922c68330c537b8` on both sides —
so DeLong (on AUROC) and McNemar (on the 0.5-threshold decisions) are the right tests. If the
Task 04-01 out-of-fold file is attached, they are computed; if not, the table still prints from the
published numbers and says which cells could not be tested.

**To attach it:** `Add Data →` the dataset holding `task04_01_oof_predictions.npz`
(from `task04_outputs.zip`), or `task05_01_oof_predictions.npz` when `MODEL = "vit_tiny"`.

In [ ]:
from scipy import stats

def delong_test(y, p1, p2):
    """Sun & Xu fast DeLong for two correlated ROC curves on the same cases."""
    y = np.asarray(y)
    pos, neg = y == 1, y == 0
    m, n = int(pos.sum()), int(neg.sum())

    def midrank(x):
        j = np.argsort(x); z = x[j]
        n_, i, r = len(x), 0, np.empty(len(x))
        while i < n_:
            k = i
            while k < n_ - 1 and z[k + 1] == z[i]:
                k += 1
            r[i:k + 1] = 0.5 * (i + k) + 1
            i = k + 1
        out = np.empty(n_); out[j] = r
        return out

    preds = np.vstack([p1, p2])
    tx, ty = preds[:, pos], preds[:, neg]
    tz = np.hstack([tx, ty])
    v01 = np.vstack([(midrank(tz[r])[:m] - midrank(tx[r])) / n for r in range(2)])
    v10 = np.vstack([1.0 - (midrank(tz[r])[m:] - midrank(ty[r])) / m for r in range(2)])
    auc = np.array([(midrank(tz[r])[:m].sum() - m * (m + 1) / 2) / (m * n) for r in range(2)])
    s = np.cov(v01) / m + np.cov(v10) / n
    var = s[0, 0] + s[1, 1] - 2 * s[0, 1]
    if var <= 0:
        return float(auc[0]), float(auc[1]), np.nan, np.nan
    z = (auc[0] - auc[1]) / np.sqrt(var)
    # norm.sf, not 1 - norm.cdf: the latter underflows to exactly 0 beyond |z| ~ 8.3
    return float(auc[0]), float(auc[1]), float(z), float(2 * stats.norm.sf(abs(z)))


def mcnemar_exact(y, p1, p2, thr=0.5):
    a, b = (p1 >= thr).astype(int) == y, (p2 >= thr).astype(int) == y
    n01, n10 = int((~a & b).sum()), int((a & ~b).sum())
    if n01 + n10 == 0:
        return n01, n10, 1.0
    return n01, n10, float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)


def find_baseline_oof(cfg):
    """Locate the matching Task 04-01 / 05-01 out-of-fold file. Fingerprint must agree."""
    want = "task04_01_oof_predictions.npz" if cfg["MODEL"] == "resnet18" \
        else "task05_01_oof_predictions.npz"
    for r in ["/kaggle/input", cfg["WORK"], "."]:
        for p in sorted(glob.glob(f"{r}/**/{want}", recursive=True)):
            d = np.load(p, allow_pickle=True)
            fp = str(d["fingerprint"])
            if fp != FINGERPRINT:
                print(f"  skipping {p}: fingerprint {fp} != {FINGERPRINT}")
                continue
            order = np.argsort(d["cache_index"])
            return p, d["p"][order], d["y"][order].astype(np.int64)
    return None, None, None


BASE_PATH, BASE_P, BASE_Y = find_baseline_oof(CFG)
if BASE_PATH:
    assert np.array_equal(BASE_Y, LABELS), "the attached baseline has different labels"
    print(f"paired baseline: {BASE_PATH}  (fingerprint matches)")
    M_BASE = core_metrics(LABELS, BASE_P)
else:
    print("No matching baseline out-of-fold file attached - the paired tests are skipped.")
    print("Attach the dataset containing task04_01_oof_predictions.npz to enable them.")
    M_BASE = None

In [ ]:
# Published Task 04-01 / 05-01 numbers on the v2 cache, so the table is complete even when the
# out-of-fold file is not attached. Used only for display; every test below uses the attached file.
PUBLISHED = {
    "resnet18": dict(auroc=0.8833, ci=[0.845, 0.919], ece=0.1372, fn=25,
                     label="Task 04-01  ResNet-18, full fine-tuning"),
    "vit_tiny": dict(auroc=0.6923, ci=[0.626, 0.756], ece=0.0693, fn=40,
                     label="Task 05-01  ViT-tiny, full fine-tuning"),
}
pub = PUBLISHED[CFG["MODEL"]]

rows = [dict(variant=pub["label"], role="BASELINE (headline)",
             trainable_params="11.2M" if CFG["MODEL"] == "resnet18" else "5.5M",
             auroc=(M_BASE["auroc"] if M_BASE else pub["auroc"]),
             auroc_ci=f"[{pub['ci'][0]:.3f}-{pub['ci'][1]:.3f}]",
             ece=(M_BASE["ece"] if M_BASE else pub["ece"]),
             fn=(M_BASE["fn"] if M_BASE else pub["fn"]),
             sens_at_spec90=(M_BASE["sens_at_spec_90"] if M_BASE else np.nan),
             hpo="none (fixed a priori)")]

def head_size(params):
    return sum(q.numel() for q in ClassifierHead(
        FEAT_DIM, params["n_layers"], params["hidden"], params["dropout"],
        params["act"], params["use_bn"]).parameters())

for tag, m, params in [("frozen backbone + untuned linear probe", M_DEFAULT, "default"),
                       ("frozen backbone + Optuna-tuned head", M_TUNED, "tuned")]:
    # the tuned head has a different shape in every fold, so report the range rather than fold 0
    sizes = [head_size(DEFAULT_PARAMS)] if params == "default" else             [head_size(BEST_PARAMS[k]) for k in range(CFG["N_FOLDS"])]
    span = f"{min(sizes)/1e3:.1f}K" if min(sizes) == max(sizes) else            f"{min(sizes)/1e3:.1f}-{max(sizes)/1e3:.1f}K"
    rows.append(dict(variant=tag, role="ablation",
                     trainable_params=span,
                     auroc=m["auroc"], auroc_ci=f"[{m['auroc_ci'][0]:.3f}-{m['auroc_ci'][1]:.3f}]",
                     ece=m["ece"], fn=m["fn"], sens_at_spec90=m["sens_at_spec_90"],
                     hpo="none" if params == "default"
                         else f"Optuna, {CFG['N_TRIALS']}x{CFG['N_FOLDS']} trials, nested"))

ABLATION = pd.DataFrame(rows)
print("ABLATION TABLE  |  fingerprint " + FINGERPRINT + "  |  n = " + str(len(LABELS)))
print(ABLATION.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

PAIRED = []
if M_BASE is not None:
    for name, p in [("frozen_default", OOF_DEFAULT), ("frozen_tuned", OOF_TUNED)]:
        a1, a2, z, pv = delong_test(LABELS, BASE_P, p)
        n01, n10, pm = mcnemar_exact(LABELS, BASE_P, p)
        PAIRED.append(dict(comparison=f"baseline vs {name}", auroc_a=a1, auroc_b=a2,
                           delta=a1 - a2, delong_z=z, delong_p=pv,
                           mcnemar_b_only=n01, mcnemar_a_only=n10, mcnemar_p=pm))
    a1, a2, z, pv = delong_test(LABELS, OOF_TUNED, OOF_DEFAULT)
    n01, n10, pm = mcnemar_exact(LABELS, OOF_TUNED, OOF_DEFAULT)
    PAIRED.append(dict(comparison="frozen_tuned vs frozen_default", auroc_a=a1, auroc_b=a2,
                       delta=a1 - a2, delong_z=z, delong_p=pv,
                       mcnemar_b_only=n01, mcnemar_a_only=n10, mcnemar_p=pm))
    print("\nPAIRED TESTS (same nodules, same folds)")
    print(pd.DataFrame(PAIRED).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("\ndelta > 0 means the first model has the higher AUROC.")
else:
    a1, a2, z, pv = delong_test(LABELS, OOF_TUNED, OOF_DEFAULT)
    PAIRED.append(dict(comparison="frozen_tuned vs frozen_default", auroc_a=a1, auroc_b=a2,
                       delta=a1 - a2, delong_z=z, delong_p=pv))
    print("\nOnly the within-notebook comparison is available:")
    print(pd.DataFrame(PAIRED).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### 7.3 &nbsp;Reading the table honestly

Two questions the ablation is there to answer, and the shape of the answer in each case:

**Does the Optuna search help?** That is `frozen_tuned vs frozen_default`. Both use frozen
features, so the only difference is the head and its optimisation. This is the comparison that
speaks directly to Dr. Sattar's request, and it is the one with the cleanest interpretation.

**Do frozen features match full fine-tuning?** That is `baseline vs frozen_*`. On 322 nodules a
gap of a few AUROC points will usually have a confidence interval wide enough to include zero.
If DeLong does not reject, the correct statement is *"frozen features were not shown to differ
from fine-tuning at this sample size"*, not *"frozen features are as good"* — these are different
claims and only the first is supported.

There is one asymmetry worth stating in the paper: the frozen arm had a 200-trial hyper-parameter
search and the baseline had none, since Task 04-01 fixed everything a priori (its audit A5). If
the frozen arm wins, some of that margin is bought by the search, not by the representation. That
cuts against the frozen arm being read as "better", which is a further reason to keep it where
constraint (c) puts it — as an ablation row.

## 8 &nbsp;What freezing costs the uncertainty story

Constraint (c) ends with the point that matters most for the paper: *"our contribution is
uncertainty-aware and agentic inference, so please do not chase a higher number at the cost of that
story."* There is a concrete mechanism behind that warning, and it is worth recording here rather
than discovering it in Task 6.

MC Dropout estimates epistemic uncertainty by sampling **stochastic layers**. In the Task 04-01
network there are five of them spread through the backbone — `d1`–`d4` after each residual stage,
plus the head dropout — so a forward pass perturbs the *representation*, and the spread across
passes reflects genuine disagreement about what the image contains. That is the mechanism behind
Task 6's result for ResNet: ECE 0.1372 → 0.0796, with an error-ranking AUROC of 0.767.

In a frozen model the backbone is deterministic. Every MC sample computes the **same** feature
vector, and only the head's dropout varies. Epistemic uncertainty then measures how sensitive a
small classifier is to dropping units in its own layers — a much narrower question than how
sensitive the representation is to the image. This is the same structural problem found in Task 5,
where timm's `drop_rate` wired dropout only into the classifier and left 50 block dropouts at
`p = 0.0`; fixing it there required `proj_drop_rate` precisely because head-only dropout was not
enough.

So the frozen variant is cheap to train and cheap to search, and it is a legitimate ablation row —
but it is not a suitable backbone for the uncertainty-aware inference that is the actual
contribution. The head checkpoints are still saved with dropout in place, so Task 6 *can* be run
against this arm; the expectation, stated in advance, is that its epistemic component will be
compressed relative to the fine-tuned network. That is a prediction this notebook makes, not a
result it reports.

## 9 &nbsp;Figures

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))
arms = [("baseline (fine-tuned)", BASE_P, "#1f3b73"),
        ("frozen + default head", OOF_DEFAULT, "#4a8ec2"),
        ("frozen + Optuna head", OOF_TUNED, "#c2564a")]
arms = [a for a in arms if a[1] is not None]

for name, p, c in arms:
    fpr, tpr, _ = roc_curve(LABELS, p)
    ax[0].plot(fpr, tpr, color=c, lw=2, label=f"{name}  {roc_auc_score(LABELS, p):.3f}")
ax[0].plot([0, 1], [0, 1], "k--", lw=.8)
ax[0].set_xlabel("1 - specificity"); ax[0].set_ylabel("sensitivity")
ax[0].set_title("Out-of-fold ROC"); ax[0].legend(fontsize=8, loc="lower right")

for k in range(CFG["N_FOLDS"]):
    v = TRIALS_DF[(TRIALS_DF.fold == k) & TRIALS_DF.inner_cv_auroc.notna()]
    if len(v):
        ax[1].plot(v.trial, v.inner_cv_auroc.cummax(), lw=1.6, label=f"fold {k}")
ax[1].set_xlabel("trial"); ax[1].set_ylabel("best inner-CV AUROC so far")
ax[1].set_title(f"Optuna search ({len(TRIALS_DF)} trials)"); ax[1].legend(fontsize=8)

ok = TRIALS_DF[TRIALS_DF.inner_cv_auroc.notna()]
ax[2].scatter(ok.lr, ok.inner_cv_auroc, s=14, c=ok.fold, cmap="viridis", alpha=.75)
ax[2].set_xscale("log"); ax[2].set_xlabel("learning rate"); ax[2].set_ylabel("inner-CV AUROC")
ax[2].set_title("Objective vs learning rate")

bins = np.linspace(0, 1, CFG["N_ECE_BINS"] + 1)
for name, p, c in arms:
    mid, obs = [], []
    for i in range(len(bins) - 1):
        m = (p >= bins[i]) & (p < bins[i + 1]) if i < len(bins) - 2 else \
            (p >= bins[i]) & (p <= bins[i + 1])
        if m.sum() >= 3:
            mid.append(p[m].mean()); obs.append(LABELS[m].mean())
    ax[3].plot(mid, obs, "o-", color=c, ms=4, lw=1.4, label=name)
ax[3].plot([0, 1], [0, 1], "k--", lw=.8)
ax[3].set_xlabel("predicted probability"); ax[3].set_ylabel("observed frequency")
ax[3].set_title("Calibration"); ax[3].legend(fontsize=7)

banner = "  [SMOKE - SYNTHETIC DATA, NOT REPORTABLE]" if IS_SMOKE else ""
fig.suptitle(f"Task 10 - frozen {CFG['MODEL']} + Optuna  |  fingerprint {FINGERPRINT}  |  "
             f"n = {len(LABELS)}{banner}", y=1.03)
plt.tight_layout()
plt.savefig(f"{CFG['OUT_DIR']}/task10_{CFG['MODEL']}_results.png", dpi=150, bbox_inches="tight")
plt.show()

## 10 &nbsp;Outputs

Seven files, each of which is used by something downstream: the paper's ablation row, the paired
tests, the reported search space and budget, or a possible Task 6 run against this arm. Nothing
else is written.

In [ ]:
OUT, M = CFG["OUT_DIR"], CFG["MODEL"]

np.savez_compressed(
    f"{OUT}/task10_{M}_oof_predictions.npz",
    p_frozen_default=OOF_DEFAULT, p_frozen_tuned=OOF_TUNED,
    y=LABELS, cache_index=np.arange(len(LABELS)), patient_id=PATIENT_ID,
    fold=FOLD_OF, malignancy=MALIGNANCY, model=M, frozen=True,
    fingerprint=FINGERPRINT, build_version=BUILD_VERSION)

TRIALS_DF.to_csv(f"{OUT}/task10_{M}_optuna_trials.csv", index=False)
ABLATION.to_csv(f"{OUT}/task10_{M}_ablation_table.csv", index=False)

json.dump({"search_space": SEARCH_SPACE,
           "trials_per_outer_fold": CFG["N_TRIALS"],
           "outer_folds": CFG["N_FOLDS"],
           "inner_folds": CFG["INNER_FOLDS"],
           "total_trials": int(len(TRIALS_DF)),
           "total_head_fits": int(len(TRIALS_DF) * CFG["INNER_FOLDS"]),
           "sampler": "TPESampler(multivariate=True)",
           "pruner": "MedianPruner(n_startup_trials=8, n_warmup_steps=1)",
           "objective": f"mean AUROC over {CFG['INNER_FOLDS']} patient-grouped inner folds "
                        f"of the outer training portion",
           "max_epochs": CFG["HEAD_MAX_EPOCHS"], "patience": CFG["HEAD_PATIENCE"],
           "selected_per_fold": {str(k): BEST_PARAMS[k] for k in BEST_PARAMS},
           "untuned_reference": DEFAULT_PARAMS,
           "search_wall_clock_min": round(SEARCH_MIN, 2), "device": DEVICE.type},
          open(f"{OUT}/task10_{M}_search_space.json", "w"), indent=2, default=str)

SUMMARY = {
    "task": f"Task 10 - frozen {M} feature extractor + Optuna head search",
    "role": "ABLATION - not a replacement for the Task 04-01 baseline (constraint c)",
    "model": M, "backbone_frozen": True,
    "classification_layer_removed": BACKBONE.removed,
    "feature_dim": FEAT_DIM,
    "build_version": BUILD_VERSION, "fingerprint": FINGERPRINT,
    "fingerprint_matches_task04_05": bool(FINGERPRINT == EXPECTED_FINGERPRINT),
    "smoke": bool(IS_SMOKE), "n_nodules": int(len(LABELS)),
    "n_patients": int(len(np.unique(PATIENT_ID))),
    "protocol": {k: CFG[k] for k in PROTOCOL_KEYS},
    "constraint_a_nested_search": {
        "objective": "inner-CV AUROC on the training portion only",
        # counted from TOUCHED_BY_SEARCH, which accumulated during the run - see Audit A8
        "nodules_read_by_search_outside_train": int(sum(
            len(TOUCHED_BY_SEARCH[k] - set(f["train"].tolist())) for k, f in enumerate(FOLDS))),
        "test_nodules_read_by_search": int(sum(
            len(TOUCHED_BY_SEARCH[k] & set(f["test"].tolist())) for k, f in enumerate(FOLDS))),
        "validation_nodules_read_by_search": int(sum(
            len(TOUCHED_BY_SEARCH[k] & set(f["val"].tolist())) for k, f in enumerate(FOLDS)))},
    "pooled_oof": {"frozen_default": M_DEFAULT, "frozen_tuned": M_TUNED,
                   "baseline_attached": M_BASE},
    "per_fold": {"frozen_default": FOLDS_DEFAULT.to_dict("records"),
                 "frozen_tuned": FOLDS_TUNED.to_dict("records")},
    "paired_tests": PAIRED,
    "ablation_table": ABLATION.to_dict("records"),
    "uncertainty_caveat": ("MC Dropout on a frozen backbone samples only the head; the epistemic "
                           "component is expected to be compressed relative to Task 04-01. See §8."),
    "checkpoints": {"default": CKPT_DEFAULT, "tuned": CKPT_TUNED},
    "environment": {"torch": torch.__version__, "optuna": optuna.__version__,
                    "device": DEVICE.type, "gpu": GPU_OK},
}
json.dump(SUMMARY, open(f"{OUT}/task10_{M}_summary.json", "w"), indent=2, default=str)

for f in sorted(os.listdir(OUT)):
    print(f"  {os.path.getsize(f'{OUT}/{f}')/1e3:9.1f} KB  {f}")

### 10.1 &nbsp;Download everything as one zip

Run this last. The zip lands in `/kaggle/working`; open the **Output** panel on the right and
download `task10_outputs.zip`.

In [ ]:
zip_path = f"{CFG['WORK']}/task10_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(CFG["OUT_DIR"]):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, CFG["WORK"]))

print(f"{zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("\nOutput panel (right) -> task10_outputs.zip -> download.")
print(f"\nheadline: frozen {CFG['MODEL']} + Optuna head, out-of-fold AUROC "
      f"{M_TUNED['auroc']:.4f} (untuned probe {M_DEFAULT['auroc']:.4f})")
print("role    : ablation row. The Task 04-01 baseline remains the headline model.")
if FINGERPRINT == EXPECTED_FINGERPRINT:
    print(f"fingerprint {FINGERPRINT} - unchanged, so Task 5 does not need rerunning.")
else:
    print(f"fingerprint {FINGERPRINT} != expected {EXPECTED_FINGERPRINT} - this run is NOT")
    print("comparable with Task 04-01 / 05-01, and per constraint (b) Task 5 would need rerunning.")